# 🟣 Chapter 5: Model Evaluation and Improvement
**Referensi:** *Introduction to Machine Learning with Python* (Andreas C. Müller & Sarah Guido)
---
> **Author:** Dhafi Dzakwan Pratama | 101032300213
> **Topik:** Cross-Validation, Grid Search, dan Evaluation Metrics
---
## 1. Pendahuluan
Mengevaluasi model hanya dengan *train_test_split* bisa menipu karena hasilnya sangat bergantung pada bagaimana data diacak. Bab ini membahas metode evaluasi yang lebih kuat, serta cara mencari parameter terbaik untuk model secara otomatis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split

%matplotlib inline

## 2. Cross-Validation
Dalam *K-Fold Cross-Validation*, data dibagi menjadi $K$ bagian (fold). Model dilatih $K$ kali, di mana setiap kalinya satu bagian berbeda digunakan sebagai data uji dan sisanya sebagai data latih. Ini memberikan estimasi performa model yang lebih stabil.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

iris = load_iris()
logreg = LogisticRegression(max_iter=1000)

# Melakukan 5-Fold Cross Validation
scores = cross_val_score(logreg, iris.data, iris.target, cv=5)

print("Skor cross-validation:", scores)
print(f"Rata-rata akurasi: {scores.mean():.2f}")

# Visualisasi kestabilan akurasi di setiap fold
plt.figure(figsize=(8, 4))
plt.bar(range(1, 6), scores, color='skyblue', edgecolor='black')
plt.axhline(scores.mean(), color='red', linestyle='--', label='Rata-rata')
plt.ylim(0.8, 1.05)
plt.xlabel('Fold ke-')
plt.ylabel('Akurasi')
plt.title('Akurasi Logistic Regression pada 5-Fold Cross-Validation')
plt.legend();

## 3. Grid Search
Mencari *hyperparameter* terbaik (seperti nilai `C` dan `gamma` pada SVM) secara manual sangat merepotkan. `GridSearchCV` secara otomatis mencoba semua kombinasi parameter yang kita tentukan dan mencari yang menghasilkan akurasi tertinggi.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Menentukan parameter grid yang akan dicoba
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100],
              'gamma': [0.001, 0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(SVC(), param_grid, cv=5, return_train_score=True)
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, random_state=0)

# Menjalankan grid search (mencoba 36 kombinasi C dan gamma)
grid_search.fit(X_train, y_train)

print(f"Parameter terbaik: {grid_search.best_params_}")
print(f"Skor cross-validation terbaik: {grid_search.best_score_:.2f}")

# Visualisasi Hasil Grid Search (Heatmap)
results = pd.DataFrame(grid_search.cv_results_)
scores_matrix = np.array(results.mean_test_score).reshape(6, 6)

plt.figure(figsize=(8, 6))
plt.imshow(scores_matrix, interpolation='nearest', cmap='viridis')
plt.xlabel('gamma')
plt.ylabel('C')
plt.colorbar(label='Mean Test Score')
plt.xticks(np.arange(6), param_grid['gamma'])
plt.yticks(np.arange(6), param_grid['C'])
plt.title('Akurasi Grid Search SVM (Heatmap)');

## 4. Evaluation Metrics (Confusion Matrix)
Untuk dataset yang tidak seimbang (*imbalanced*), akurasi bukanlah metrik yang bagus. Kita membutuhkan **Confusion Matrix** untuk melihat seberapa banyak model salah menebak kelas positif sebagai negatif (False Negative) atau sebaliknya (False Positive).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression

cancer = load_breast_cancer()
Xc_train, Xc_test, yc_train, yc_test = train_test_split(cancer.data, cancer.target, random_state=0)

logreg = LogisticRegression(max_iter=10000).fit(Xc_train, yc_train)
pred_logreg = logreg.predict(Xc_test)

# Membuat confusion matrix
cm = confusion_matrix(yc_test, pred_logreg)

# Visualisasi Confusion Matrix menggunakan sklearn modern
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cancer.target_names)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix: Prediksi Kanker Payudara");